In [1]:
import pandas as pd
import numpy as np

FILE = "D:/Tushar/Copy of Master Data _290102026 2 - Copy.xlsx"

df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

# Clean Category
df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)


In [2]:
df = df[df["Category"].isin(["repeater", "stranger"])].copy()


In [3]:
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]
    

In [4]:
T120_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

grp = df.groupby("Child Part", sort=False)

parts = grp.agg({
    "effective_daily_demand": "sum",
    "Inventory_25": "first",
    "Minimum Quantity": "first",
    "Cycle Time": "first",
    "Vertical Machines": lambda x: [
        m.strip()
        for m in ",".join(x.dropna().astype(str)).split(",")
        if m.strip() in T120_MACHINES
    ],
    "Category": "first"
}).reset_index()


In [5]:
parts = parts[parts["Vertical Machines"].map(len) > 0].copy()


In [6]:
parts.rename(columns={
    "Child Part": "part",
    "effective_daily_demand": "daily_demand",
    "Inventory_25": "inventory",
    "Minimum Quantity": "min_qty",
    "Cycle Time": "cycle_time",
    "Vertical Machines": "machines"
}, inplace=True)


In [7]:
parts["net_required_qty"] = (
    parts["daily_demand"]
    + parts["min_qty"]
    - parts["inventory"]
).clip(lower=0)


In [8]:
rows = []

for _, r in parts.iterrows():
    for m in r["machines"]:
        rows.append({
            "part": r["part"],
            "category": r["Category"],
            "machine": m,
            "daily_demand": r["daily_demand"],
            "inventory": r["inventory"],
            "cycle_time": r["cycle_time"],
            "net_required_qty": r["net_required_qty"]
        })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("No Repeater/Stranger parts found for 120T machines")


ValueError: No Repeater/Stranger parts found for 120T machines

In [ ]:
rows = []

for _, r in parts.iterrows():
    for m in r["machines"]:
        rows.append({
            "part": r["part"],
            "category": r["Category"],
            "machine": m,
            "daily_demand": r["daily_demand"],
            "inventory": r["inventory"],
            "cycle_time": r["cycle_time"],
            "net_required_qty": r["net_required_qty"]
        })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("No Repeater/Stranger parts found for 120T machines")


In [ ]:
CHANGEOVER = 40
CAPACITY = 1320
TARGET_DAYS = 3

def compute_score(row):
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    prod_time = qty_if_made * row["cycle_time"]

    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)


In [ ]:
dfm["score"] = dfm.apply(compute_score, axis=1)


In [ ]:
selected = []

for m, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(3))

selected = pd.concat(selected).reset_index(drop=True)


In [ ]:
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]


In [ ]:
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)


In [9]:
import pandas as pd
import numpy as np

# =========================================================
# CONFIG
# =========================================================
FILE = "D:/Tushar/Copy of Master Data _290102026 2 - Copy.xlsx"

T120_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

CHANGEOVER = 40          # minutes
CAPACITY = 1320          # minutes per machine per day
TARGET_DAYS = 3
MAX_PARTS_PER_MACHINE = 3

# =========================================================
# STEP 1: LOAD & CLEAN DATA
# =========================================================
df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Only Repeater & Stranger
df = df[df["Category"].isin(["repeater", "stranger"])].copy()

# =========================================================
# STEP 2: EFFECTIVE DAILY DEMAND
# =========================================================
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]

# =========================================================
# STEP 3: PART-LEVEL AGGREGATION (NO MACHINES HERE)
# =========================================================
part_level = (
    df.groupby("Child Part", sort=False)
      .agg({
          "effective_daily_demand": "sum",
          "Inventory_25": "first",
          "Minimum Quantity": "first",
          "Cycle Time": "first",
          "Category": "first"
      })
      .reset_index()
)

# =========================================================
# STEP 4: NET REQUIRED QTY (YOUR LOGIC)
# =========================================================
part_level["net_required_qty"] = (
    part_level["effective_daily_demand"]
    + part_level["Minimum Quantity"]
    - part_level["Inventory_25"]
).clip(lower=0)

# =========================================================
# STEP 5: BUILD PART–MACHINE TABLE (REAL DATA GRAIN)
# =========================================================
rows = []

for _, r in df.iterrows():
    machine = str(r["Vertical Machines"]).strip()

    if machine not in T120_MACHINES:
        continue

    p = part_level.loc[
        part_level["Child Part"] == r["Child Part"]
    ].iloc[0]

    rows.append({
        "part": r["Child Part"],
        "category": r["Category"],
        "machine": machine,
        "daily_demand": p["effective_daily_demand"],
        "inventory": p["Inventory_25"],
        "cycle_time": p["Cycle Time"],
        "net_required_qty": p["net_required_qty"]
    })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("❌ No Repeater/Stranger parts found on 120T machines")

print("✅ dfm created:", dfm.shape)

# =========================================================
# STEP 6: SCORE FUNCTION (HUMAN LOGIC)
# =========================================================
def compute_score(row):
    # Inventory pain
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    # Relief potential
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    # Production time
    prod_time = qty_if_made * row["cycle_time"]

    # Setup penalty
    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    # Monopoly penalty
    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)

dfm["score"] = dfm.apply(compute_score, axis=1)

# =========================================================
# STEP 7: SELECT TOP 2–3 PARTS PER MACHINE
# =========================================================
selected = []

for machine, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(MAX_PARTS_PER_MACHINE))

selected = pd.concat(selected).reset_index(drop=True)

# =========================================================
# STEP 8: QUANTITY TO PRODUCE (3-DAY INVENTORY)
# =========================================================
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]

# =========================================================
# FINAL OUTPUT
# =========================================================
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== FINAL DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)

ValueError: ❌ No Repeater/Stranger parts found on 120T machines

In [10]:
import pandas as pd
import numpy as np

# =========================================================
# CONFIG
# =========================================================
FILE = "D:/Tushar/Copy of Master Data _290102026 2 - Copy.xlsx"

# Human-known 120T machines
RAW_120T_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

# Normalize machine names → MP01, MP05, etc.
T120_MACHINES = {m.replace("-", "").upper() for m in RAW_120T_MACHINES}

CHANGEOVER = 40          # minutes
CAPACITY = 1320          # minutes/day
TARGET_DAYS = 3
MAX_PARTS_PER_MACHINE = 3

# =========================================================
# STEP 1: LOAD DATA
# =========================================================
df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

# =========================================================
# STEP 2: CLEAN CATEGORY
# =========================================================
df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Keep only Repeater & Stranger
df = df[df["Category"].isin(["repeater", "stranger"])].copy()

# =========================================================
# STEP 3: CLEAN MACHINE NAMES (CRITICAL FIX)
# =========================================================
df["machine_clean"] = (
    df["Vertical Machines"]
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", "", regex=True)   # remove spaces/newlines
    .str.replace("-", "", regex=False)     # remove hyphens
)

print("DEBUG → Unique cleaned machines:")
print(sorted(df["machine_clean"].unique()))

# =========================================================
# STEP 4: EFFECTIVE DAILY DEMAND
# =========================================================
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]

# =========================================================
# STEP 5: PART-LEVEL AGGREGATION (NO MACHINES)
# =========================================================
part_level = (
    df.groupby("Child Part", sort=False)
      .agg({
          "effective_daily_demand": "sum",
          "Inventory_25": "first",
          "Minimum Quantity": "first",
          "Cycle Time": "first",
          "Category": "first"
      })
      .reset_index()
)

# =========================================================
# STEP 6: NET REQUIRED QTY (YOUR LOGIC)
# =========================================================
part_level["net_required_qty"] = (
    part_level["effective_daily_demand"]
    + part_level["Minimum Quantity"]
    - part_level["Inventory_25"]
).clip(lower=0)

# =========================================================
# STEP 7: BUILD PART–MACHINE TABLE (ROW-LEVEL, FIXED)
# =========================================================
rows = []

for _, r in df.iterrows():
    machine = r["machine_clean"]

    if machine not in T120_MACHINES:
        continue

    p = part_level.loc[
        part_level["Child Part"] == r["Child Part"]
    ].iloc[0]

    rows.append({
        "part": r["Child Part"],
        "category": r["Category"],
        "machine": machine,  # normalized
        "daily_demand": p["effective_daily_demand"],
        "inventory": p["Inventory_25"],
        "cycle_time": p["Cycle Time"],
        "net_required_qty": p["net_required_qty"]
    })

dfm = pd.DataFrame(rows)

print("\nDEBUG → dfm shape:", dfm.shape)
print(dfm.head())

if dfm.empty:
    raise ValueError(
        "❌ STILL EMPTY: Check machine names printed above. "
        "They must match MP01, MP05, MP10, MP17"
    )

# =========================================================
# STEP 8: SCORE FUNCTION (HUMAN LOGIC)
# =========================================================
def compute_score(row):
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    prod_time = qty_if_made * row["cycle_time"]

    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)

dfm["score"] = dfm.apply(compute_score, axis=1)

# =========================================================
# STEP 9: SELECT MAX 3 PARTS PER MACHINE
# =========================================================
selected = []

for machine, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(MAX_PARTS_PER_MACHINE))

selected = pd.concat(selected).reset_index(drop=True)

# =========================================================
# STEP 10: QUANTITY TO PRODUCE (3-DAY INVENTORY LOGIC)
# =========================================================
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]

# =========================================================
# FINAL OUTPUT
# =========================================================
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== FINAL DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)

DEBUG → Unique cleaned machines:
['M.P01', 'M.P03', 'M.P04', 'M.P05', 'M.P07', 'M.P08', 'M.P09', 'M.P10', 'M.P13', 'M.P14', 'M.P16', 'M.P17', 'NAN', 'TOYO1ST', 'TOYO2ND', 'TOYO6TH', 'TOYO7TH', 'TOYO80T1ST', 'TOYO80T2ND']

DEBUG → dfm shape: (0, 0)
Empty DataFrame
Columns: []
Index: []


ValueError: ❌ STILL EMPTY: Check machine names printed above. They must match MP01, MP05, MP10, MP17